In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load & Clean Dataset
df = pd.read_csv('deap_processed.csv')
print(f"Raw shape: {df.shape}")

for col in ['HR', 'HRV', 'Respiration']:
    before = len(df)
    df = df[df[col] != 0]
    removed = before - len(df)
    if removed > 0:
        print(f"Removed {removed} rows with {col}=0")

df = df.reset_index(drop=True)
print(f"Clean shape: {df.shape}")

Raw shape: (1280, 6)
Removed 4 rows with Respiration=0
Clean shape: (1276, 6)


In [3]:
# Feature Scaling & Train-Test Split
X = df[['HR', 'HRV', 'Respiration']].values
y = df[['Valence', 'Arousal', 'Dominance']].values

# Scale features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Features:     {X_train.shape[1]}")
print(f"Targets:      {y_train.shape[1]} (Valence, Arousal, Dominance)")

Training set: 1020 samples
Test set:     256 samples
Features:     3
Targets:      3 (Valence, Arousal, Dominance)


In [4]:
# Model Evaluation Function
def evaluate_model(model, X_tr, y_tr, X_te, y_te, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    results = {}
    target_names = ['Valence', 'Arousal', 'Dominance']
    for i, t in enumerate(target_names):
        r2 = r2_score(y_te[:, i], y_pred[:, i])
        mae = mean_absolute_error(y_te[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_te[:, i], y_pred[:, i]))
        results[t] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    avg = {k: np.mean([results[t][k] for t in target_names]) for k in ['R2', 'MAE', 'RMSE']}
    results['Average'] = avg
    print(f"{name}")
    display(pd.DataFrame(results).round(4).T)
    return results, y_pred

all_results = {}

In [5]:
# Train All ML Models
all_results['Linear Regression'], _ = evaluate_model(
    LinearRegression(), X_train, y_train, X_test, y_test, "Linear Regression (Baseline)")

all_results['Random Forest'], _ = evaluate_model(
    RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    X_train, y_train, X_test, y_test, "Random Forest")

all_results['SVR'], _ = evaluate_model(
    MultiOutputRegressor(SVR(kernel='rbf', C=10, epsilon=0.1)),
    X_train, y_train, X_test, y_test, "Support Vector Regression")

all_results['Gradient Boosting'], _ = evaluate_model(
    MultiOutputRegressor(GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)),
    X_train, y_train, X_test, y_test, "Gradient Boosting")

all_results['KNN'], _ = evaluate_model(
    KNeighborsRegressor(n_neighbors=7, weights='distance'),
    X_train, y_train, X_test, y_test, "K-Nearest Neighbors")

Linear Regression (Baseline)


,R2,MAE,RMSE
Valence,-0.0049,1.7041,2.0114
Arousal,-0.0020,1.5139,1.8635
Dominance,0.0616,1.6760,2.0200
Average,0.0182,1.6313,1.9650


Random Forest


,R2,MAE,RMSE
Valence,-0.0311,1.6862,2.0375
Arousal,-0.0535,1.5582,1.9108
Dominance,0.0485,1.6953,2.0340
Average,-0.0120,1.6466,1.9941


Support Vector Regression


,R2,MAE,RMSE
Valence,-0.0568,1.6866,2.0627
Arousal,-0.0403,1.5150,1.8988
Dominance,0.0054,1.7159,2.0796
Average,-0.0306,1.6392,2.0137


Gradient Boosting


,R2,MAE,RMSE
Valence,-0.1967,1.7879,2.1951
Arousal,-0.1373,1.6206,1.9854
Dominance,-0.0036,1.7040,2.0890
Average,-0.1125,1.7042,2.0898


K-Nearest Neighbors


,R2,MAE,RMSE
Valence,-0.1340,1.7039,2.1368
Arousal,-0.2270,1.6852,2.0621
Dominance,-0.1119,1.7805,2.1989
Average,-0.1576,1.7232,2.1326


In [6]:
# Neural Network Training & Evaluation
scaler_y = MinMaxScaler()
y_train_nn = scaler_y.fit_transform(y_train)
y_test_nn = scaler_y.transform(y_test)

nn = MLPRegressor(
    hidden_layer_sizes=(64, 128, 64), activation='relu', solver='adam',
    learning_rate='adaptive', learning_rate_init=0.001, max_iter=500,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=30,
    batch_size=32, random_state=42, verbose=False
)
nn.fit(X_train, y_train_nn)
print(f"NN trained for {nn.n_iter_} iterations | Best validation: {nn.best_validation_score_:.4f}")

y_pred_nn = scaler_y.inverse_transform(nn.predict(X_test))
target_names = ['Valence', 'Arousal', 'Dominance']
nn_results = {}
for i, t in enumerate(target_names):
    nn_results[t] = {
        'R2': r2_score(y_test[:, i], y_pred_nn[:, i]),
        'MAE': mean_absolute_error(y_test[:, i], y_pred_nn[:, i]),
        'RMSE': np.sqrt(mean_squared_error(y_test[:, i], y_pred_nn[:, i]))
    }
nn_results['Average'] = {k: np.mean([nn_results[t][k] for t in target_names]) for k in ['R2', 'MAE', 'RMSE']}
print("Neural Network (MLP)")
display(pd.DataFrame(nn_results).round(4).T)

NN trained for 41 iterations | Best validation: -0.0070
Neural Network (MLP)


,R2,MAE,RMSE
Valence,-0.0440,1.7085,2.0502
Arousal,-0.0363,1.5616,1.8952
Dominance,0.0659,1.6759,2.0154
Average,-0.0048,1.6486,1.9869


In [7]:
# Final Comparison
all_results_final = {**all_results, 'Neural Network': nn_results}

comparison_final = {}
for m in all_results_final:
    row = {t: all_results_final[m][t]['R2'] for t in target_names}
    row['Avg R2'] = np.mean(list(row.values()))
    comparison_final[m] = row

print("Final Model Comparison - R2 Scores")
display(pd.DataFrame(comparison_final).round(4).T)

best_ml = max(all_results.keys(), key=lambda m: np.mean([all_results[m][t]['R2'] for t in target_names]))
print(f"Best ML Model: {best_ml}")

Final Model Comparison - R2 Scores


,Valence,Arousal,Dominance,Avg R2
Linear Regression,-0.0049,-0.0020,0.0616,0.0182
Random Forest,-0.0311,-0.0535,0.0485,-0.0120
SVR,-0.0568,-0.0403,0.0054,-0.0306
Gradient Boosting,-0.1967,-0.1373,-0.0036,-0.1125
KNN,-0.1340,-0.2270,-0.1119,-0.1576
Neural Network,-0.0440,-0.0363,0.0659,-0.0048


Best ML Model: Linear Regression
